# Taller 2 - Punto 1: Embeddings de Palabras con Word2vec y FastText

Este notebook implementa el entrenamiento y evaluación de modelos de embeddings de palabras usando Word2vec y FastText sobre el dataset spanish_billion_words.

## Estrategia de Experimentación:
- Tres tamaños de dataset: 500k, 5M y 10M textos
- Modelos: Word2Vec y FastText
- Dimensiones de embeddings: 100, 200, 300
- Total de experimentos: 18 (2 modelos x 3 dimensiones x 3 tamaños de dataset)
- Resultados guardados en JSON para análisis posterior
- Visualizaciones y modelos organizados en directorios separados

## 1.1 Instalación de Dependencias y Configuración Inicial

In [1]:
import nltk
nltk.download('stopwords')

from datasets import load_dataset
from nltk.corpus import stopwords
from tqdm.auto import tqdm
import re
import string
from gensim.models import FastText, Word2Vec
import numpy as np
import warnings
import time
import gc
warnings.filterwarnings('ignore')

from helpers import (
    save_model, 
    save_experiment_results, 
    load_experiment_results,
    visualize_embeddings_tsne,
    visualize_embeddings_pca,
    query_similar_words,
    print_experiment_summary,
    clear_model_from_memory,
    ensure_directories
)

ensure_directories(1)

[nltk_data] Downloading package stopwords to /home/yenreh/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
/home/yenreh/anaconda3/envs/pln_taller2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
EJECUTAR_TODOS = True
# Configuración optimizada: procesa datasets bajo demanda en lugar de todos a la vez
PROCESAR_BAJO_DEMANDA = True  # Si es True, carga datasets solo cuando se necesitan

## 1.2 Carga del Dataset Spanish Billion Words

Cargamos el dataset y preparamos las muestras de 500k, 5M y 10M para experimentación.

In [3]:
import os

CACHE_DIR = "./input/jhonparra18"
os.makedirs(CACHE_DIR, exist_ok=True)

print("Cargando dataset spanish_billion_words...")
dataset_full = load_dataset(
    "jhonparra18/spanish_billion_words_clean", 
    split="train",
    cache_dir=CACHE_DIR
)

print(f"Dataset completo cargado: {len(dataset_full):,} textos")
print(f"Cache guardado en: {CACHE_DIR}")

Cargando dataset spanish_billion_words...
Dataset completo cargado: 46,925,295 textos
Cache guardado en: ./input/jhonparra18
Dataset completo cargado: 46,925,295 textos
Cache guardado en: ./input/jhonparra18


## 1.3 Preprocesamiento del Corpus

Función de limpieza del texto que se aplicará a cada muestra del dataset.

In [4]:
def preprocess_text(dataset_sample):
    """
    Preprocesa el texto eliminando números, stopwords, puntuación y símbolos especiales
    Usa multiprocesamiento para mayor velocidad
    """
    from helpers import preprocess_text_parallel
    
    stop_words = stopwords.words('spanish')
    sentences = preprocess_text_parallel(dataset_sample, stop_words)
    
    return sentences

## 1.3.1 Preprocesamiento de Datasets

Estrategia optimizada de memoria: Los datasets se pueden preprocesar bajo demanda durante los experimentos, 
o pre-procesarlos todos aquí. El modo bajo demanda ahorra memoria al cargar solo lo necesario.

In [5]:
from helpers import save_preprocessed_sentences, load_preprocessed_sentences

if PROCESAR_BAJO_DEMANDA:
    print("="*80)
    print("MODO DE PROCESAMIENTO BAJO DEMANDA ACTIVADO")
    print("="*80)
    
    # Solo inicializamos las variables como None
    sentences_500k = None
    sentences_5m = None
    sentences_10m = None
else:
    print("="*80)
    print("PREPROCESANDO TODOS LOS DATASETS")
    print("="*80)

    print("\n1. Preparando muestra de 500,000 textos...")
    sentences_500k = load_preprocessed_sentences("sentences_500k", punto=1)

    if sentences_500k is None:
        print("Cache no encontrado. Procesando desde cero...")
        dataset_500k = dataset_full.select(range(min(500000, len(dataset_full))))
        sentences_500k = preprocess_text(dataset_500k)
        save_preprocessed_sentences(sentences_500k, "sentences_500k", punto=1)
    else:
        print("Dataset 500k cargado desde cache")

    print("\n2. Preparando muestra de 5,000,000 textos...")
    sentences_5m = load_preprocessed_sentences("sentences_5m", punto=1)
    
    if sentences_5m is None:
        print("Cache no encontrado. Procesando dataset 5M desde cero...")
        dataset_5m = dataset_full.select(range(min(5000000, len(dataset_full))))
        sentences_5m = preprocess_text(dataset_5m)
        save_preprocessed_sentences(sentences_5m, "sentences_5m", punto=1)
    else:
        print("Dataset 5M cargado desde cache")

    print("\n3. Preparando muestra de 10,000,000 textos...")
    sentences_10m = load_preprocessed_sentences("sentences_10m", punto=1)
    
    if sentences_10m is None:
        print("Cache no encontrado. Procesando dataset 10M desde cero...")
        dataset_10m = dataset_full.select(range(min(10000000, len(dataset_full))))
        sentences_10m = preprocess_text(dataset_10m)
        save_preprocessed_sentences(sentences_10m, "sentences_10m", punto=1)
    else:
        print("Dataset 10M cargado desde cache")

    print("\n" + "="*80)
    print("DATASETS PREPROCESADOS Y LISTOS")
    print("="*80)
    print(f"  500k: {len(sentences_500k):,} sentencias")
    print(f"  5M:   {len(sentences_5m):,} sentencias")
    print(f"  10M:  {len(sentences_10m):,} sentencias")
    print("="*80)

MODO DE PROCESAMIENTO BAJO DEMANDA ACTIVADO


## 1.4 Configuración de Experimentos

Definimos todos los experimentos a ejecutar con diferentes combinaciones de modelos, dimensiones y tamaños de dataset.

In [6]:
experiments = [
    # Word2Vec - 500k
    {'model_type': 'word2vec', 'vector_size': 100, 'dataset_size': 500000, 'dataset_name': '500k'},
    {'model_type': 'word2vec', 'vector_size': 200, 'dataset_size': 500000, 'dataset_name': '500k'},
    {'model_type': 'word2vec', 'vector_size': 300, 'dataset_size': 500000, 'dataset_name': '500k'},
    
    # Word2Vec - 5M
    {'model_type': 'word2vec', 'vector_size': 100, 'dataset_size': 5000000, 'dataset_name': '5m'},
    {'model_type': 'word2vec', 'vector_size': 200, 'dataset_size': 5000000, 'dataset_name': '5m'},
    {'model_type': 'word2vec', 'vector_size': 300, 'dataset_size': 5000000, 'dataset_name': '5m'},
    
    # Word2Vec - 10M
    {'model_type': 'word2vec', 'vector_size': 100, 'dataset_size': 10000000, 'dataset_name': '10m'},
    {'model_type': 'word2vec', 'vector_size': 200, 'dataset_size': 10000000, 'dataset_name': '10m'},
    {'model_type': 'word2vec', 'vector_size': 300, 'dataset_size': 10000000, 'dataset_name': '10m'},
    
    # FastText - 500k
    {'model_type': 'fasttext', 'vector_size': 100, 'dataset_size': 500000, 'dataset_name': '500k'},
    {'model_type': 'fasttext', 'vector_size': 200, 'dataset_size': 500000, 'dataset_name': '500k'},
    {'model_type': 'fasttext', 'vector_size': 300, 'dataset_size': 500000, 'dataset_name': '500k'},
    
    # FastText - 5M
    {'model_type': 'fasttext', 'vector_size': 100, 'dataset_size': 5000000, 'dataset_name': '5m'},
    {'model_type': 'fasttext', 'vector_size': 200, 'dataset_size': 5000000, 'dataset_name': '5m'},
    {'model_type': 'fasttext', 'vector_size': 300, 'dataset_size': 5000000, 'dataset_name': '5m'},
    
    # FastText - 10M
    {'model_type': 'fasttext', 'vector_size': 100, 'dataset_size': 10000000, 'dataset_name': '10m'},
    {'model_type': 'fasttext', 'vector_size': 200, 'dataset_size': 10000000, 'dataset_name': '10m'},
    {'model_type': 'fasttext', 'vector_size': 300, 'dataset_size': 10000000, 'dataset_name': '10m'},
]

print(f"Total de experimentos configurados: {len(experiments)}")
for i, exp in enumerate(experiments, 1):
    print(f"{i:2}. {exp['model_type']:10} - {exp['vector_size']}d - {exp['dataset_name']:5}")

Total de experimentos configurados: 18
 1. word2vec   - 100d - 500k 
 2. word2vec   - 200d - 500k 
 3. word2vec   - 300d - 500k 
 4. word2vec   - 100d - 5m   
 5. word2vec   - 200d - 5m   
 6. word2vec   - 300d - 5m   
 7. word2vec   - 100d - 10m  
 8. word2vec   - 200d - 10m  
 9. word2vec   - 300d - 10m  
10. fasttext   - 100d - 500k 
11. fasttext   - 200d - 500k 
12. fasttext   - 300d - 500k 
13. fasttext   - 100d - 5m   
14. fasttext   - 200d - 5m   
15. fasttext   - 300d - 5m   
16. fasttext   - 100d - 10m  
17. fasttext   - 200d - 10m  
18. fasttext   - 300d - 10m  


## 1.5 Función de Entrenamiento y Evaluación

Función unificada que ejecuta un experimento completo: entrenamiento, evaluación, visualización y guardado de resultados.
OPTIMIZACIÓN DE MEMORIA: Carga datasets bajo demanda y libera memoria después de cada experimento.

In [7]:
def load_dataset_on_demand(dataset_size, dataset_name, dataset_full):
    """
    Carga un dataset específico bajo demanda desde cache o lo procesa
    """
    from helpers import load_preprocessed_sentences, save_preprocessed_sentences
    
    cache_name = f"sentences_{dataset_name}"
    
    print(f"Cargando dataset {dataset_name} ({dataset_size:,} textos)...")
    sentences = load_preprocessed_sentences(cache_name, punto=1)
    
    if sentences is None:
        print(f"Cache no encontrado. Procesando dataset {dataset_name} desde cero...")
        dataset_sample = dataset_full.select(range(min(dataset_size, len(dataset_full))))
        sentences = preprocess_text(dataset_sample)
        save_preprocessed_sentences(sentences, cache_name, punto=1)
        print(f"Dataset procesado y guardado en cache")
        
        # Liberar memoria del dataset temporal
        del dataset_sample
        gc.collect()
    else:
        print(f"Dataset cargado desde cache")
    
    print(f"Dataset {dataset_name} listo: {len(sentences):,} sentencias")
    return sentences


def run_experiment(exp_config, sentences_500k, sentences_5m, sentences_10m, dataset_full):
    """
    Ejecuta un experimento completo de entrenamiento y evaluación
    OPTIMIZACIÓN: Carga datasets bajo demanda si no están en memoria
    """
    model_type = exp_config['model_type']
    vector_size = exp_config['vector_size']
    dataset_size = exp_config['dataset_size']
    dataset_name = exp_config['dataset_name']
    
    experiment_name = f"{model_type}_{vector_size}d_{dataset_name}"
    
    print("\n" + "="*80)
    print(f"EXPERIMENTO: {experiment_name}")
    print("="*80)
    
    # Selección inteligente de dataset con carga bajo demanda
    dataset_loaded_here = False
    
    if dataset_size == 500000:
        if sentences_500k is None:
            sentences = load_dataset_on_demand(dataset_size, dataset_name, dataset_full)
            dataset_loaded_here = True
        else:
            sentences = sentences_500k
            print(f"Usando dataset 500k pre-cargado: {len(sentences):,} sentencias")
            
    elif dataset_size == 5000000:
        if sentences_5m is None:
            sentences = load_dataset_on_demand(dataset_size, dataset_name, dataset_full)
            dataset_loaded_here = True
        else:
            sentences = sentences_5m
            print(f"Usando dataset 5M pre-cargado: {len(sentences):,} sentencias")
            
    elif dataset_size == 10000000:
        if sentences_10m is None:
            sentences = load_dataset_on_demand(dataset_size, dataset_name, dataset_full)
            dataset_loaded_here = True
        else:
            sentences = sentences_10m
            print(f"Usando dataset 10M pre-cargado: {len(sentences):,} sentencias")
    else:
        print(f"ERROR: Tamaño de dataset no reconocido: {dataset_size}")
        return None
    
    print(f"\nEntrenando modelo {model_type.upper()} con vector_size={vector_size}...")
    start_time = time.time()
    
    if model_type == 'word2vec':
        model = Word2Vec(
            sentences=sentences,
            vector_size=vector_size,
            window=5,
            min_count=10,
            workers=4,
            sg=0
        )
    else:
        model = FastText(
            sentences=sentences,
            vector_size=vector_size,
            window=5,
            min_count=10,
            workers=4,
            sg=0
        )
    
    training_time = time.time() - start_time
    vocab_size = len(model.wv.index_to_key)
    
    print(f"Entrenamiento completado en {training_time:.2f} segundos")
    print(f"Vocabulario: {vocab_size:,} palabras")
    
    model_path = save_model(model, experiment_name, punto=1)
    print(f"Modelo guardado: {model_path}")
    
    test_word = 'futuro'
    print(f"\nConsultando similitud para '{test_word}'...")
    similar_words_result = query_similar_words(model, test_word, topn=5)
    
    if 'similar_words' in similar_words_result:
        for sw in similar_words_result['similar_words']:
            print(f"  {sw['word']}: {sw['similarity']:.4f}")
    
    print(f"\nGenerando visualización t-SNE...")
    tsne_path = visualize_embeddings_tsne(model, experiment_name, punto=1, num_words=100)
    
    print(f"\nGenerando visualización PCA...")
    pca_path, variance_explained = visualize_embeddings_pca(model, experiment_name, punto=1, num_words=100)
    
    results = {
        'model_type': model_type,
        'vector_size': vector_size,
        'dataset_name': dataset_name,
        'dataset_size': len(sentences),
        'vocab_size': vocab_size,
        'training_time': training_time,
        'variance_explained': variance_explained,
        'model_path': model_path,
        'visualizations': [tsne_path, pca_path],
        'similar_words_sample': similar_words_result
    }
    
    results_file = save_experiment_results(punto=1, experiment_name=experiment_name, results=results)
    print(f"\nResultados guardados en: {results_file}")
    
    print(f"\nLiberando memoria...")
    clear_model_from_memory(model)
    
    # Si cargamos el dataset bajo demanda, también lo liberamos
    if dataset_loaded_here:
        del sentences
        print(f"Dataset temporal liberado de memoria")
    
    gc.collect()
    
    print(f"\nEXPERIMENTO {experiment_name} COMPLETADO")
    print("="*80)
    
    return results

## 1.6 Ejecución de Experimentos

Ejecutamos todos los experimentos configurados usando los datasets ya preprocesados.

In [8]:
if EJECUTAR_TODOS:
    experimentos_a_ejecutar = experiments
else:
    experimentos_a_ejecutar = [exp for exp in experiments if exp['dataset_name'] == '500k'][:3]

print(f"Se ejecutarán {len(experimentos_a_ejecutar)} experimentos")
if PROCESAR_BAJO_DEMANDA:
    print("Modo: CARGA BAJO DEMANDA (optimizado para memoria)")
else:
    print("Modo: DATASETS PRE-CARGADOS")
print()

for i, exp in enumerate(experimentos_a_ejecutar, 1):
    print(f"\n{'#'*80}")
    print(f"# EXPERIMENTO {i}/{len(experimentos_a_ejecutar)}")
    print(f"{'#'*80}")
    
    try:
        result = run_experiment(exp, sentences_500k, sentences_5m, sentences_10m, dataset_full)
        if result is None:
            print("Experimento omitido")
            continue
    except Exception as e:
        print(f"\nERROR en experimento {exp}: {e}")
        import traceback
        traceback.print_exc()
        continue

print("\n" + "="*80)
print("TODOS LOS EXPERIMENTOS COMPLETADOS")
print("="*80)

Se ejecutarán 18 experimentos
Modo: CARGA BAJO DEMANDA (optimizado para memoria)


################################################################################
# EXPERIMENTO 1/18
################################################################################

EXPERIMENTO: word2vec_100d_500k
Cargando dataset 500k (500,000 textos)...
Cargando sentencias desde cache: ./input/preprocessed/punto1/sentences_500k.pkl
Cargadas 499,895 sentencias
Dataset cargado desde cache
Dataset 500k listo: 499,895 sentencias

Entrenando modelo WORD2VEC con vector_size=100...
Cargadas 499,895 sentencias
Dataset cargado desde cache
Dataset 500k listo: 499,895 sentencias

Entrenando modelo WORD2VEC con vector_size=100...
Entrenamiento completado en 10.93 segundos
Vocabulario: 39,530 palabras
Modelo guardado: ./models/punto1/word2vec_100d_500k.model

Consultando similitud para 'futuro'...
  desafío: 0.7409
  reto: 0.7372
  progreso: 0.7171
  lograr: 0.7064
  estrategia: 0.7032

Generando visualización t-SN

## 1.7 Visualización de Resultados

Imprimimos un resumen de todos los experimentos ejecutados desde el archivo JSON.

In [9]:
print_experiment_summary(punto=1)


RESUMEN DE EXPERIMENTOS - PUNTO 1

Experimento 1: word2vec_100d_500k
Timestamp: 2025-11-16T22:16:30.550130
--------------------------------------------------------------------------------
Tipo de modelo: word2vec
Dimensión de embeddings: 100
Tamaño del dataset: 499,895 sentencias
Vocabulario: 39,530 palabras
Tiempo de entrenamiento: 10.76 segundos
Varianza explicada (PCA): 17.81%

Ejemplo de similitud para 'futuro':
  reto: 0.7184
  lograr: 0.7129
  compromiso: 0.7014

Modelo guardado en: ./models/punto1/word2vec_100d_500k.model
Visualizaciones:
  ./output/punto1/word2vec_100d_500k_tsne.png
  ./output/punto1/word2vec_100d_500k_pca.png

Experimento 2: word2vec_200d_500k
Timestamp: 2025-11-16T22:17:00.989507
--------------------------------------------------------------------------------
Tipo de modelo: word2vec
Dimensión de embeddings: 200
Tamaño del dataset: 499,895 sentencias
Vocabulario: 39,530 palabras
Tiempo de entrenamiento: 14.14 segundos
Varianza explicada (PCA): 16.55%

Ejempl

## 1.8 Análisis Comparativo Detallado

Análisis estadístico y comparativo de los resultados obtenidos.

In [10]:
import pandas as pd

results = load_experiment_results(punto=1)

if results:
    df = pd.DataFrame(results)
    
    print("\n" + "="*80)
    print("ANÁLISIS COMPARATIVO DE EXPERIMENTOS")
    print("="*80)
    
    print("\n1. TIEMPOS DE ENTRENAMIENTO:")
    for model_type in df['model_type'].unique():
        subset = df[df['model_type'] == model_type]
        print(f"\n   {model_type.upper()}:")
        for _, row in subset.iterrows():
            print(f"     {row['vector_size']}d - {row['dataset_name']:5}: {row['training_time']:7.2f}s")
    
    print("\n2. VARIANZA EXPLICADA (PCA):")
    for model_type in df['model_type'].unique():
        subset = df[df['model_type'] == model_type]
        print(f"\n   {model_type.upper()}:")
        for _, row in subset.iterrows():
            print(f"     {row['vector_size']}d - {row['dataset_name']:5}: {row['variance_explained']*100:5.2f}%")
    
    print("\n3. TAMAÑOS DE VOCABULARIO:")
    for model_type in df['model_type'].unique():
        subset = df[df['model_type'] == model_type]
        print(f"\n   {model_type.upper()}:")
        for _, row in subset.iterrows():
            print(f"     {row['vector_size']}d - {row['dataset_name']:5}: {row['vocab_size']:,} palabras")
    
    print("\n4. COMPARACIÓN WORD2VEC vs FASTTEXT:")
    if len(df[df['model_type'] == 'word2vec']) > 0 and len(df[df['model_type'] == 'fasttext']) > 0:
        w2v_avg_time = df[df['model_type'] == 'word2vec']['training_time'].mean()
        ft_avg_time = df[df['model_type'] == 'fasttext']['training_time'].mean()
        w2v_avg_var = df[df['model_type'] == 'word2vec']['variance_explained'].mean()
        ft_avg_var = df[df['model_type'] == 'fasttext']['variance_explained'].mean()
        
        print(f"\n   Tiempo promedio de entrenamiento:")
        print(f"     Word2Vec: {w2v_avg_time:.2f}s")
        print(f"     FastText: {ft_avg_time:.2f}s")
        print(f"\n   Varianza explicada promedio (PCA):")
        print(f"     Word2Vec: {w2v_avg_var*100:.2f}%")
        print(f"     FastText: {ft_avg_var*100:.2f}%")
    
    print("\n5. MEJOR CONFIGURACIÓN POR MÉTRICA:")
    best_variance = df.loc[df['variance_explained'].idxmax()]
    fastest = df.loc[df['training_time'].idxmin()]
    largest_vocab = df.loc[df['vocab_size'].idxmax()]
    
    print(f"\n   Mayor varianza explicada:")
    print(f"     {best_variance['experiment_name']}: {best_variance['variance_explained']*100:.2f}%")
    print(f"\n   Entrenamiento más rápido:")
    print(f"     {fastest['experiment_name']}: {fastest['training_time']:.2f}s")
    print(f"\n   Mayor vocabulario:")
    print(f"     {largest_vocab['experiment_name']}: {largest_vocab['vocab_size']:,} palabras")
    
    print("\n" + "="*80)
else:
    print("No hay resultados disponibles. Ejecute los experimentos primero.")


ANÁLISIS COMPARATIVO DE EXPERIMENTOS

1. TIEMPOS DE ENTRENAMIENTO:

   WORD2VEC:
     100d - 500k :   10.76s
     200d - 500k :   14.14s
     300d - 500k :   16.97s
     100d - 500k :   10.93s
     200d - 500k :   14.45s
     300d - 500k :   17.35s
     100d - 5m   :  135.14s
     200d - 5m   :  165.68s
     300d - 5m   :  200.18s
     100d - 10m  :  278.27s
     200d - 10m  :  370.04s
     300d - 10m  :  436.80s

   FASTTEXT:
     100d - 500k :   39.17s
     200d - 500k :   60.64s
     300d - 500k :   76.95s
     100d - 5m   :  439.54s
     200d - 5m   :  678.81s
     300d - 5m   :  863.72s
     100d - 10m  : 1017.03s
     200d - 10m  : 1600.24s
     300d - 10m  : 2021.10s

2. VARIANZA EXPLICADA (PCA):

   WORD2VEC:
     100d - 500k : 17.81%
     200d - 500k : 16.55%
     300d - 500k : 16.38%
     100d - 500k : 17.78%
     200d - 500k : 16.47%
     300d - 500k : 16.39%
     100d - 5m   : 16.35%
     200d - 5m   : 13.88%
     300d - 5m   : 12.65%
     100d - 10m  : 16.98%
     200d - 

## 1.9 Conclusiones

Conclusiones finales basadas en los experimentos ejecutados.

In [11]:
results = load_experiment_results(punto=1)

print("\n" + "="*80)
print("CONCLUSIONES - PUNTO 1: EMBEDDINGS DE PALABRAS")
print("="*80)

if results:
    df = pd.DataFrame(results)
    
    print(f"\n1. EXPERIMENTOS REALIZADOS:")
    print(f"   Total de experimentos: {len(results)}")
    print(f"   Modelos evaluados: {', '.join(df['model_type'].unique())}")
    print(f"   Dimensiones probadas: {sorted(df['vector_size'].unique())}")
    print(f"   Tamaños de dataset: {', '.join(df['dataset_name'].unique())}")
    
    print(f"\n2. CORPUS Y PREPROCESAMIENTO:")
    max_dataset = df['dataset_size'].max()
    min_dataset = df['dataset_size'].min()
    print(f"   Dataset completo: {max_dataset:,} sentencias procesadas")
    print(f"   Dataset muestra: {min_dataset:,} sentencias procesadas")
    avg_vocab = df['vocab_size'].mean()
    print(f"   Vocabulario promedio: {avg_vocab:,.0f} palabras únicas")
    
    print(f"\n3. RENDIMIENTO:")
    print(f"   Tiempo de entrenamiento (promedio): {df['training_time'].mean():.2f}s")
    print(f"   Tiempo mínimo: {df['training_time'].min():.2f}s")
    print(f"   Tiempo máximo: {df['training_time'].max():.2f}s")
    
    print(f"\n4. CALIDAD DE EMBEDDINGS:")
    print(f"   Varianza explicada (PCA promedio): {df['variance_explained'].mean()*100:.2f}%")
    print(f"   Mejor varianza explicada: {df['variance_explained'].max()*100:.2f}%")
    
    if 'word2vec' in df['model_type'].values and 'fasttext' in df['model_type'].values:
        w2v_var = df[df['model_type'] == 'word2vec']['variance_explained'].mean()
        ft_var = df[df['model_type'] == 'fasttext']['variance_explained'].mean()
        
        print(f"\n5. COMPARACIÓN WORD2VEC vs FASTTEXT:")
        print(f"   Word2Vec:")
        print(f"     - Varianza promedio: {w2v_var*100:.2f}%")
        print(f"     - Mejor para: Relaciones semánticas puras")
        print(f"   FastText:")
        print(f"     - Varianza promedio: {ft_var*100:.2f}%")
        print(f"     - Mejor para: Manejo de palabras OOV y variaciones morfológicas")
    
    print(f"\n6. RECOMENDACIONES:")
    best = df.loc[df['variance_explained'].idxmax()]
    print(f"   Configuración óptima: {best['experiment_name']}")
    print(f"   - Tipo: {best['model_type'].upper()}")
    print(f"   - Dimensión: {best['vector_size']}")
    print(f"   - Dataset: {best['dataset_name']}")
    print(f"   - Varianza explicada: {best['variance_explained']*100:.2f}%")
    
    print(f"\n7. ARCHIVOS GENERADOS:")
    print(f"   Modelos guardados en: ./models/punto1/")
    print(f"   Visualizaciones en: ./output/punto1/")
    print(f"   Resultados JSON en: ./results/punto1_results.json")
else:
    print("\nNo hay resultados disponibles.")
    print("Ejecute los experimentos en la sección 1.6 primero.")

print("\n" + "="*80)


CONCLUSIONES - PUNTO 1: EMBEDDINGS DE PALABRAS

1. EXPERIMENTOS REALIZADOS:
   Total de experimentos: 21
   Modelos evaluados: word2vec, fasttext
   Dimensiones probadas: [np.int64(100), np.int64(200), np.int64(300)]
   Tamaños de dataset: 500k, 5m, 10m

2. CORPUS Y PREPROCESAMIENTO:
   Dataset completo: 9,996,928 sentencias procesadas
   Dataset muestra: 499,895 sentencias procesadas
   Vocabulario promedio: 125,822 palabras únicas

3. RENDIMIENTO:
   Tiempo de entrenamiento (promedio): 403.23s
   Tiempo mínimo: 10.76s
   Tiempo máximo: 2021.10s

4. CALIDAD DE EMBEDDINGS:
   Varianza explicada (PCA promedio): 18.64%
   Mejor varianza explicada: 25.87%

5. COMPARACIÓN WORD2VEC vs FASTTEXT:
   Word2Vec:
     - Varianza promedio: 15.87%
     - Mejor para: Relaciones semánticas puras
   FastText:
     - Varianza promedio: 22.33%
     - Mejor para: Manejo de palabras OOV y variaciones morfológicas

6. RECOMENDACIONES:
   Configuración óptima: fasttext_100d_10m
   - Tipo: FASTTEXT
   - Dim